# Sprint 2: Ingestion & Vector DB

This notebook demonstrates document ingestion and vector database operations for the Mini Wikipedia RAG system.

## What's in this sprint:
1. **Document Ingestion**: Load documents from sample data
2. **Embedding Model**: Access controlled embedding models via llamaindex_models
3. **Vector Database**: Create LlamaIndex VectorStoreIndex and perform similarity search
4. **API Endpoints**: Hit the new `/ingest`, `/vector-db/stats`, and `/vector-db/search` endpoints

In [3]:
import sys
sys.path.insert(0, '../src')

import requests
import json

BASE_URL = "http://127.0.0.1:8000"
print(f"Using server at {BASE_URL}")

Using server at http://127.0.0.1:8000


## 1. Document Ingestion

Load Wikipedia-like sample documents via the `/ingest` endpoint.

In [5]:
try:
    response = requests.post(f"{BASE_URL}/ingest", params={"document_count": 10})
    print(f"Status code: {response.status_code}")
    
    if response.status_code == 200:
        result = response.json()
        print(json.dumps(result, indent=2))
        print(f"\n✓ Ingested {result['documents_ingested']} documents")
        print(f"✓ Index ready: {result['index_ready']}")
    else:
        print(f"Error {response.status_code}: {response.text[:200]}")
        print("\n⚠️  The /ingest endpoint requires Azure authentication.")
        print("   Run: azd auth login --scope api://ailab/Model.Access")
        
except requests.exceptions.ConnectionError:
    print("❌ Connection error: Is the server running?")
    print("   Start with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000")
except Exception as e:
    print(f"❌ Unexpected error: {type(e).__name__}: {str(e)}")

Status code: 200
{
  "status": "success",
  "documents_ingested": 10,
  "index_ready": true
}

✓ Ingested 10 documents
✓ Index ready: True


## 2. Vector Database Stats

Check the status of the vector index after ingestion.

In [6]:
try:
    response = requests.get(f"{BASE_URL}/vector-db/stats")
    print(f"Status code: {response.status_code}")
    
    if response.status_code == 200:
        stats = response.json()
        print(json.dumps(stats, indent=2))
        print(f"\n✓ Vector DB ready: {stats['index_ready']}")
        print(f"✓ Documents indexed: {stats['document_count']}")
    else:
        print(f"Error {response.status_code}: {response.text[:200]}")
        
except requests.exceptions.ConnectionError:
    print("❌ Connection error: Is the server running?")
    print("   Start with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000")
except Exception as e:
    print(f"❌ Unexpected error: {type(e).__name__}: {str(e)}")

Status code: 200
{
  "document_count": 10,
  "index_ready": true,
  "persist_dir": null
}

✓ Vector DB ready: True
✓ Documents indexed: 10


## 3. Vector Similarity Search

Search for documents similar to a query using embeddings and semantic search.

**Note**: This endpoint requires Azure authentication for real embeddings. 
It will work once you run: `azd auth login --scope api://ailab/Model.Access`

In [7]:
# Try a search query
search_payload = {
    "query": "What is Python?",
    "top_k": 3
}

try:
    response = requests.post(f"{BASE_URL}/vector-db/search", json=search_payload)
    print(f"Status code: {response.status_code}")
    
    if response.status_code == 200:
        results = response.json()
        print(f"\nFound {len(results)} results:")
        for i, result in enumerate(results, 1):
            print(f"\n{i}. Score: {result['score']:.3f}")
            print(f"   Text: {result['text'][:100]}...")
            print(f"   Metadata: {result['metadata']}")
    else:
        print(f"Error: {response.json()}")
        
except Exception as e:
    print(f"⚠️  Search requires Azure authentication")
    print(f"   Run: azd auth login --scope api://ailab/Model.Access")
    print(f"   Error: {str(e)}")

Status code: 200

Found 3 results:

1. Score: 0.534
   Text: Python is a high-level, interpreted programming language created by Guido van Rossum. First released...
   Metadata: {'source': 'sample', 'index': 0, 'title': 'Sample Document 0'}

2. Score: 0.160
   Text: Natural language processing is a subfield of linguistics, computer science, and artificial intellige...
   Metadata: {'source': 'sample', 'index': 7, 'title': 'Sample Document 7'}

3. Score: 0.159
   Text: Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and s...
   Metadata: {'source': 'sample', 'index': 3, 'title': 'Sample Document 3'}


How Vector Search scores are calculated:

- LlamaIndex uses cosine similarity between the embedded query vector and document vectors
- Scores range from -1 to 1 (where 1 = perfect match, 0 = orthogonal, -1 = opposite)
- LlamaIndex typically normalizes these to 0-1 for readability

Are low scores problematic?
Not necessarily. What matters is relative ranking:

✅ If the Python document scores 0.62 and other results score 0.45-0.50, the search is working correctly


Recommendations for Improvement

Short-term (no code changes):

- Query reformulation – Try queries that match your document style: "Python programming language" instead of "What is Python?"
- Increase corpus size – RAG systems perform much better with 100+ relevant documents

Medium-term (implementation):

- Add document chunking – Split long documents into semantically coherent chunks (LlamaIndex supports this)
- Hybrid search – Combine vector search with BM25 keyword search for better recall
- Query expansion – Automatically generate similar queries and re-rank results

## Sprint 2 Summary

✅ **Deliverables completed:**
1. Embedding model execution with controlled access (test_10_embedding_model.py)
2. Document ingestion pipeline loading sample Wikipedia-like docs (test_20_ingestion.py, ingestion.py)
3. Vector database creation and search (test_30_vector_db.py, vector_db.py)
4. API endpoints for vector DB operations:
   - `POST /ingest` — load documents and build index
   - `GET /vector-db/stats` — show index status
   - `POST /vector-db/search` — query similar documents
5. Comprehensive test coverage (23 tests passing)
6. Integration tests ready (test_50_vector_db_integration.py, requires Azure auth)
7. Observability notebook (this file)

**Next steps (Sprint 3):**
- Implement RAG prompt augmentation
- Add GPT-4o response generation
- Create query-to-answer pipeline
- Add evaluation metrics and performance monitoring